# Regression Model Accuracy Testing
___

Import pandas and the three regression classes.

In [39]:
import pandas as pd
from ml_algorithms.regression import BivariableLinearRegression, MultipleLinearRegression, LogisticLinearRegression

Load the dataset and drop any rows with missing values.

In [40]:
df = pd.read_csv('data\StudentPerformanceFactors.csv').dropna()
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


Categorical columns need to be encoded as integers so the models can work with them. Each unique string value is mapped to a number.

In [41]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category').cat.codes

df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,1,0,0,7,73,1,1,0,1,2,1,2,3,0,1,2,1,67
1,19,64,1,2,0,8,59,1,1,2,2,2,1,0,4,0,0,1,0,61
2,24,98,2,2,1,7,91,2,1,2,2,2,1,1,4,0,2,2,1,74
3,29,89,1,2,1,8,98,2,1,1,2,2,1,0,4,0,1,1,1,71
4,19,92,2,2,1,6,65,2,1,3,2,0,1,1,4,0,0,2,0,70


Shuffle and split into 80% training and 20% testing.

In [42]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

split = int(len(df) * 0.8)
train = df.iloc[:split]
test  = df.iloc[split:]

print(f'Train: {len(train)} rows, Test: {len(test)} rows')

Train: 5102 rows, Test: 1276 rows


___
## BivariableLinearRegression
Predicting `Exam_Score` from `Hours_Studied` only.

In [61]:
model_biv = BivariableLinearRegression(
    dataset=train[['Hours_Studied', 'Exam_Score']].reset_index(drop=True),
    iterations=5000
)

predictions = model_biv.predict(test['Hours_Studied'].values)
actual      = test['Exam_Score'].values

mae  = sum(abs(predictions - actual)) / len(actual)
rmse = (sum((predictions - actual) ** 2) / len(actual)) ** 0.5

print(f'MAE  : {mae:.4f}')
print(f'RMSE : {rmse:.4f}')

MAE  : 6.4394
RMSE : 8.1327


___
## MultipleLinearRegression
Predicting `Exam_Score` from all features.

In [64]:
features = [c for c in df.columns if c != 'Exam_Score']

model_multi = MultipleLinearRegression(
    dataset=train[features + ['Exam_Score']].reset_index(drop=True),
    iterations=5000,
    learning_rate=0.00005
)

predictions = model_multi.predict(test[features].values).flatten()
actual      = test['Exam_Score'].values

mae  = sum(abs(predictions - actual)) / len(actual)
rmse = (sum((predictions - actual) ** 2) / len(actual)) ** 0.5

print(f'MAE  : {mae:.4f}')
print(f'RMSE : {rmse:.4f}')

MAE  : 3.2430
RMSE : 4.4108


___
## LogisticLinearRegression
Classifying whether a student passes (Exam_Score >= 70) or fails. `Exam_Score` is excluded from the features to avoid leaking the answer into the model.

In [82]:
df_log = df.copy()
df_log['Pass'] = (df_log['Exam_Score'] >= 70).astype(int)

log_features = [c for c in features if c != 'Exam_Score']

train_log = df_log[log_features + ['Pass']].iloc[:split]
test_log  = df_log[log_features + ['Pass']].iloc[split:]

model_log = LogisticLinearRegression(
    dataset=train_log.reset_index(drop=True),
    iterations=3000,
    learning_rate=0.1
)

predictions = model_log.predict(test_log[log_features].values).flatten()
actual      = test_log['Pass'].values

correct  = sum(predictions == actual)
accuracy = correct / len(actual) * 100

print(f'Correct : {correct} / {len(actual)}')
print(f'Accuracy: {accuracy:.1f}%')

Correct : 955 / 1276
Accuracy: 74.8%
